# Knn Known Flow Testing\n
Uses 12,000 total sampled known records: 9,600 training, 1,200 validation, 1,200 final testing, drawn from separate existing partitions.\n

In [16]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import joblib
ROOT=Path.cwd().resolve()
if ROOT.name.lower()=='knn_known' and ROOT.parent.name.lower()=='notebooks': ROOT=ROOT.parent.parent
elif ROOT.name.lower()=='notebooks': ROOT=ROOT.parent
MODULE_DIR=ROOT/'notebooks'/'knn_known'
if not (MODULE_DIR/'knn_known_flow.py').exists():
    MODULE_DIR=Path.cwd().resolve()
if str(MODULE_DIR) not in sys.path: sys.path.insert(0,str(MODULE_DIR))
from knn_known_flow import read_data, choose_columns, feature_frame, build_pipeline, save_artifact, load_artifact, predict_rows
DATA=ROOT/'data'/'processed'/'Data_V3_SSH_Unknown_final'
MODEL=ROOT/'saved-models'/'knn_known'/'known_classifier.joblib'
OUT=ROOT/'outputs'/'results'/'knn_known'
FIG=ROOT/'outputs'/'figures'/'knn_known'
OUT.mkdir(parents=True,exist_ok=True);FIG.mkdir(parents=True,exist_ok=True)
print('Root:',ROOT)
print('Model:',MODEL)

Root: C:\Users\saaral\Desktop\gc\pattern recog\project\]code\Hybrid-Network-Intrusion-Detection
Model: C:\Users\saaral\Desktop\gc\pattern recog\project\]code\Hybrid-Network-Intrusion-Detection\saved-models\knn_known\known_classifier.joblib


In [17]:
# Demo source is the unknown-attack KB-unmatched partition.
DEMO=ROOT/'data'/'processed'/'knn_kb'/'demo_similarity_audit'/'autoencoder_demo_unmatched.parquet'
if not DEMO.exists(): raise FileNotFoundError(f'Run KB demo similarity audit first: {DEMO}')
if not MODEL.exists(): raise FileNotFoundError('Run known KNN training first')
artifact=load_artifact(MODEL)
unknown=pd.read_parquet(DEMO)
unknown.columns=unknown.columns.astype(str).str.strip()
if unknown.empty: raise ValueError('No unmatched demo rows to classify')
results=predict_rows(unknown,artifact)
if 'demo_row' in unknown: results.insert(0,'demo_row',unknown['demo_row'].to_numpy())
else: results.insert(0,'demo_row',np.arange(len(unknown)))
print('Unknown demo rows:',len(results))
print('Known classes available:',artifact['classes'])
print('These are forced known-class predictions; NOT proof that unknown flows belong to those classes.')
display(results)
results.to_csv(OUT/'unknown_to_known_classification.csv',index=False)
print('Saved:',OUT/'unknown_to_known_classification.csv')

Unknown demo rows: 14
Known classes available: ['Botnet', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowloris', 'Heartbleed', 'Infiltration', 'Infiltration - Portscan', 'Portscan', 'Web Attack - Brute Force', 'Web Attack - SQL Injection', 'Web Attack - XSS']
These are forced known-class predictions; NOT proof that unknown flows belong to those classes.


,demo_row,predicted_known_class,knn_confidence,interpretation
0,79,Web Attack - Brute Force,0.336303,Known-class prediction (not proof of membership)
1,81,Infiltration,0.340912,Known-class prediction (not proof of membership)
2,134,Web Attack - XSS,0.669305,Known-class prediction (not proof of membership)
3,163,Web Attack - XSS,0.668635,Known-class prediction (not proof of membership)
4,196,Web Attack - Brute Force,0.652003,Known-class prediction (not proof of membership)
5,236,DoS Slowloris,1.000000,Known-class prediction (not proof of membership)
6,240,DoS Hulk,0.668392,Known-class prediction (not proof of membership)
7,254,Web Attack - Brute Force,0.655201,Known-class prediction (not proof of membership)
8,259,DoS Hulk,0.644078,Known-class prediction (not proof of membership)
9,272,DoS GoldenEye,0.654413,Known-class prediction (not proof of membership)


Saved: C:\Users\saaral\Desktop\gc\pattern recog\project\]code\Hybrid-Network-Intrusion-Detection\outputs\results\knn_known\unknown_to_known_classification.csv


In [15]:
# Print each flow and its known-class prediction for integration/debugging.
PRINT_FULL_FLOWS=False
for i,row in results.iterrows():
    print(f"demo_row={row['demo_row']} -> known_class={row['predicted_known_class']} confidence={row['knn_confidence']:.4f}")
    if PRINT_FULL_FLOWS:
        print(unknown.iloc[i].to_json(indent=2))

demo_row=79 -> known_class=DoS Hulk confidence=0.8543
demo_row=81 -> known_class=DDoS confidence=0.8299
demo_row=134 -> known_class=Web Attack - Brute Force confidence=0.7594
demo_row=163 -> known_class=Web Attack - Brute Force confidence=0.7113
demo_row=196 -> known_class=Web Attack - Brute Force confidence=0.8537
demo_row=236 -> known_class=DoS Slowloris confidence=1.0000
demo_row=240 -> known_class=DoS Hulk confidence=0.8089
demo_row=254 -> known_class=Web Attack - Brute Force confidence=0.9492
demo_row=259 -> known_class=DoS Hulk confidence=0.8959
demo_row=272 -> known_class=DoS GoldenEye confidence=0.6639
demo_row=335 -> known_class=Web Attack - Brute Force confidence=0.8555
demo_row=371 -> known_class=DDoS confidence=0.6631
demo_row=391 -> known_class=Web Attack - Brute Force confidence=0.8542
demo_row=432 -> known_class=DoS Hulk confidence=1.0000
